# Set up example code

## Model setup

In [1]:
import subprocess

from sklearn.gaussian_process import GaussianProcessRegressor as GPR
from sklearn.gaussian_process import kernels
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
# from sklearn.externals import joblib
import joblib
import re

import matplotlib.cm as cm
import matplotlib.pyplot as plt

from scipy.linalg import lapack
from scipy import stats
import emcee
import numpy as np

import importlib

import os
import pickle
from pathlib import Path

import src.reader as Reader

## Step 1: prepare input pickle file

### Load stuff from text files

In [2]:
DataList = ['PHENIX_200_Hadron_0To10', 'ATLAS_2760_Hadron_0To5', 'CMS_5020_Hadron_30To50',
           'STAR_200_R02_Jet_0To10', 'STAR_200_R04_Jet_0To10',
           'ALICE_2760_R02_Jet_0To10', 'ATLAS_2760_R04_Jet_0To10',
           'CMS_2760_R02_Jet_0To5', 'CMS_2760_R02_Jet_5To10',
           'CMS_2760_R03_Jet_0To5', 'CMS_2760_R03_Jet_5To10',
           'CMS_2760_R04_Jet_0To5', 'CMS_2760_R04_Jet_5To10',
           'ALICE_5020_R02_Jet_0To10', 'ALICE_5020_R04_Jet_0To10',
           'ATLAS_5020_R04_Jet_0To10', 'ATLAS_5020_R04_Jet_10To20', 'ATLAS_5020_R04_Jet_20To30',
           'ATLAS_5020_R04_Jet_30To40', 'ATLAS_5020_R04_Jet_40To50',
           'CMS_5020_R02_Jet_0To10', 'CMS_5020_R02_Jet_10To30', 'CMS_5020_R02_Jet_30To50',
           'CMS_5020_R03_Jet_0To10', 'CMS_5020_R03_Jet_10To30', 'CMS_5020_R03_Jet_30To50',
           'CMS_5020_R04_Jet_0To10', 'CMS_5020_R04_Jet_10To30', 'CMS_5020_R04_Jet_30To50',
           'CMS_5020_R06_Jet_0To10', 'CMS_5020_R06_Jet_10To30', 'CMS_5020_R06_Jet_30To50',
           'CMS_5020_R08_Jet_0To10', 'CMS_5020_R08_Jet_10To30', 'CMS_5020_R08_Jet_30To50',
           'CMS_5020_R10_Jet_10To30', 'CMS_5020_R10_Jet_30To50']

# Read data files
RawData = {}
RawData['PHENIX_200_Hadron_0To10'] = Reader.ReadData('input/STAT2022FirstTry/Data_PHENIX_AuAu200_RAACharged_0-10_2013.dat')
RawData['STAR_200_R02_Jet_0To10'] = Reader.ReadData('input/STAT2022FirstTry/Data_STAR_AuAu200_JetRAAR02_0-10_2011.dat')
RawData['STAR_200_R04_Jet_0To10'] = Reader.ReadData('input/STAT2022FirstTry/Data_STAR_AuAu200_JetRAAR04_0-10_2011.dat')
RawData['ATLAS_2760_Hadron_0To5'] = Reader.ReadData('input/STAT2022FirstTry/Data_ATLAS_PbPb2760_RAACharged_0-5_2015.dat')
RawData['CMS_5020_Hadron_30To50'] = Reader.ReadData('input/STAT2022FirstTry/Data_CMS_PbPb5020_RAACharged_30-50_2017.dat')
RawData['ALICE_2760_R02_Jet_0To10'] = Reader.ReadData('input/STAT2022FirstTry/Data_ALICE_PbPb2760_JetRAAR02_0-10_2011.dat')
RawData['ATLAS_2760_R04_Jet_0To10'] = Reader.ReadData('input/STAT2022FirstTry/Data_ATLAS_PbPb2760_JetRAAR04_0-10_2011.dat')
RawData['CMS_2760_R02_Jet_0To5'] = Reader.ReadData('input/STAT2022FirstTry/Data_CMS_PbPb2760_JetRAAR02_0-5_2011.dat')
RawData['CMS_2760_R02_Jet_5To10'] = Reader.ReadData('input/STAT2022FirstTry/Data_CMS_PbPb2760_JetRAAR02_5-10_2011.dat')
RawData['CMS_2760_R03_Jet_0To5'] = Reader.ReadData('input/STAT2022FirstTry/Data_CMS_PbPb2760_JetRAAR03_0-5_2011.dat')
RawData['CMS_2760_R03_Jet_5To10'] = Reader.ReadData('input/STAT2022FirstTry/Data_CMS_PbPb2760_JetRAAR03_5-10_2011.dat')
RawData['CMS_2760_R04_Jet_0To5'] = Reader.ReadData('input/STAT2022FirstTry/Data_CMS_PbPb2760_JetRAAR04_0-5_2011.dat')
RawData['CMS_2760_R04_Jet_5To10'] = Reader.ReadData('input/STAT2022FirstTry/Data_CMS_PbPb2760_JetRAAR04_5-10_2011.dat')
RawData['ALICE_5020_R02_Jet_0To10'] = Reader.ReadData('input/STAT2022FirstTry/Data_ALICE_PbPb5020_JetRAAR02_0-10_2015.dat')
RawData['ALICE_5020_R04_Jet_0To10'] = Reader.ReadData('input/STAT2022FirstTry/Data_ALICE_PbPb5020_JetRAAR04_0-10_2015.dat')
RawData['ATLAS_5020_R04_Jet_0To10'] = Reader.ReadData('input/STAT2022FirstTry/Data_ATLAS_PbPb5020_JetRAAR04_0-10_2015.dat')
RawData['ATLAS_5020_R04_Jet_10To20'] = Reader.ReadData('input/STAT2022FirstTry/Data_ATLAS_PbPb5020_JetRAAR04_10-20_2015.dat')
RawData['ATLAS_5020_R04_Jet_20To30'] = Reader.ReadData('input/STAT2022FirstTry/Data_ATLAS_PbPb5020_JetRAAR04_20-30_2015.dat')
RawData['ATLAS_5020_R04_Jet_30To40'] = Reader.ReadData('input/STAT2022FirstTry/Data_ATLAS_PbPb5020_JetRAAR04_30-40_2015.dat')
RawData['ATLAS_5020_R04_Jet_40To50'] = Reader.ReadData('input/STAT2022FirstTry/Data_ATLAS_PbPb5020_JetRAAR04_40-50_2015.dat')
RawData['CMS_5020_R02_Jet_0To10'] = Reader.ReadData('input/STAT2022FirstTry/Data_CMS_PbPb5020_JetRAAR02_0-10_2015.dat')
RawData['CMS_5020_R02_Jet_10To30'] = Reader.ReadData('input/STAT2022FirstTry/Data_CMS_PbPb5020_JetRAAR02_10-30_2015.dat')
RawData['CMS_5020_R02_Jet_30To50'] = Reader.ReadData('input/STAT2022FirstTry/Data_CMS_PbPb5020_JetRAAR02_30-50_2015.dat')
RawData['CMS_5020_R03_Jet_0To10'] = Reader.ReadData('input/STAT2022FirstTry/Data_CMS_PbPb5020_JetRAAR03_0-10_2015.dat')
RawData['CMS_5020_R03_Jet_10To30'] = Reader.ReadData('input/STAT2022FirstTry/Data_CMS_PbPb5020_JetRAAR03_10-30_2015.dat')
RawData['CMS_5020_R03_Jet_30To50'] = Reader.ReadData('input/STAT2022FirstTry/Data_CMS_PbPb5020_JetRAAR03_30-50_2015.dat')
RawData['CMS_5020_R04_Jet_0To10'] = Reader.ReadData('input/STAT2022FirstTry/Data_CMS_PbPb5020_JetRAAR04_0-10_2015.dat')
RawData['CMS_5020_R04_Jet_10To30'] = Reader.ReadData('input/STAT2022FirstTry/Data_CMS_PbPb5020_JetRAAR04_10-30_2015.dat')
RawData['CMS_5020_R04_Jet_30To50'] = Reader.ReadData('input/STAT2022FirstTry/Data_CMS_PbPb5020_JetRAAR04_30-50_2015.dat')
RawData['CMS_5020_R06_Jet_0To10'] = Reader.ReadData('input/STAT2022FirstTry/Data_CMS_PbPb5020_JetRAAR06_0-10_2015.dat')
RawData['CMS_5020_R06_Jet_10To30'] = Reader.ReadData('input/STAT2022FirstTry/Data_CMS_PbPb5020_JetRAAR06_10-30_2015.dat')
RawData['CMS_5020_R06_Jet_30To50'] = Reader.ReadData('input/STAT2022FirstTry/Data_CMS_PbPb5020_JetRAAR06_30-50_2015.dat')
RawData['CMS_5020_R08_Jet_0To10'] = Reader.ReadData('input/STAT2022FirstTry/Data_CMS_PbPb5020_JetRAAR08_0-10_2015.dat')
RawData['CMS_5020_R08_Jet_10To30'] = Reader.ReadData('input/STAT2022FirstTry/Data_CMS_PbPb5020_JetRAAR08_10-30_2015.dat')
RawData['CMS_5020_R08_Jet_30To50'] = Reader.ReadData('input/STAT2022FirstTry/Data_CMS_PbPb5020_JetRAAR08_30-50_2015.dat')
RawData['CMS_5020_R10_Jet_0To10'] = Reader.ReadData('input/STAT2022FirstTry/Data_CMS_PbPb5020_JetRAAR10_0-10_2015.dat')
RawData['CMS_5020_R10_Jet_10To30'] = Reader.ReadData('input/STAT2022FirstTry/Data_CMS_PbPb5020_JetRAAR10_10-30_2015.dat')
RawData['CMS_5020_R10_Jet_30To50'] = Reader.ReadData('input/STAT2022FirstTry/Data_CMS_PbPb5020_JetRAAR10_30-50_2015.dat')

# Read covariance
# RawCov1 = Reader.ReadCovariance('input/Example/Covariance_PHENIX_AuAu200_RAACharged_0to10_2013_PHENIX_AuAu200_RAACharged_0to10_2013_SmallL.dat')

# Read design points
RawDesign = Reader.ReadDesign('input/STAT2022FirstTry/Design40.dat')
RawDesign["Design"][:, 2] = np.log(RawDesign["Design"][:, 2])
RawDesign["Design"][:, 3] = np.log(RawDesign["Design"][:, 3])
RawDesign["Design"][:, 5] = np.log(RawDesign["Design"][:, 5])

# Read model prediction
RawPrediction = {}
RawPrediction['PHENIX_200_Hadron_0To10'] = Reader.ReadPrediction('input/STAT2022FirstTry/Prediction_phenix_AuAu200exponential_hadron_pt_pi0_0to10.dat')
RawPrediction['STAR_200_R02_Jet_0To10'] = Reader.ReadPrediction('input/STAT2022FirstTry/Prediction_star_AuAu200exponential_inclusive_chjet_pt_R0.2_0to10.dat')
RawPrediction['STAR_200_R04_Jet_0To10'] = Reader.ReadPrediction('input/STAT2022FirstTry/Prediction_star_AuAu200exponential_inclusive_chjet_pt_R0.4_0to10.dat')
RawPrediction['ATLAS_2760_Hadron_0To5'] = Reader.ReadPrediction('input/STAT2022FirstTry/Prediction_atlas_PbPb2760exponential_hadron_pt_ch_0to5.dat')
RawPrediction['CMS_5020_Hadron_30To50'] = Reader.ReadPrediction('input/STAT2022FirstTry/Prediction_cms_PbPb5020exponential_hadron_pt_ch_30to50.dat')
RawPrediction['ALICE_2760_R02_Jet_0To10'] = Reader.ReadPrediction('input/STAT2022FirstTry/Prediction_alice_PbPb2760exponential_inclusive_jet_pt_R0.2_0to10.dat')
RawPrediction['ATLAS_2760_R04_Jet_0To10'] = Reader.ReadPrediction('input/STAT2022FirstTry/Prediction_atlas_PbPb2760exponential_inclusive_jet_pt_R0.4_0to10.dat')
RawPrediction['CMS_2760_R02_Jet_0To5'] = Reader.ReadPrediction('input/STAT2022FirstTry/Prediction_cms_PbPb2760exponential_inclusive_jet_pt_R0.2_0to5.dat')
RawPrediction['CMS_2760_R02_Jet_5To10'] = Reader.ReadPrediction('input/STAT2022FirstTry/Prediction_cms_PbPb2760exponential_inclusive_jet_pt_R0.2_5to10.dat')
RawPrediction['CMS_2760_R03_Jet_0To5'] = Reader.ReadPrediction('input/STAT2022FirstTry/Prediction_cms_PbPb2760exponential_inclusive_jet_pt_R0.3_0to5.dat')
RawPrediction['CMS_2760_R03_Jet_5To10'] = Reader.ReadPrediction('input/STAT2022FirstTry/Prediction_cms_PbPb2760exponential_inclusive_jet_pt_R0.3_5to10.dat')
RawPrediction['CMS_2760_R04_Jet_0To5'] = Reader.ReadPrediction('input/STAT2022FirstTry/Prediction_cms_PbPb2760exponential_inclusive_jet_pt_R0.4_0to5.dat')
RawPrediction['CMS_2760_R04_Jet_5To10'] = Reader.ReadPrediction('input/STAT2022FirstTry/Prediction_cms_PbPb2760exponential_inclusive_jet_pt_R0.4_5to10.dat')
RawPrediction['ALICE_5020_R02_Jet_0To10'] = Reader.ReadPrediction('input/STAT2022FirstTry/Prediction_alice_PbPb5020exponential_inclusive_jet_pt_R0.2_0to10.dat')
RawPrediction['ALICE_5020_R04_Jet_0To10'] = Reader.ReadPrediction('input/STAT2022FirstTry/Prediction_alice_PbPb5020exponential_inclusive_jet_pt_R0.4_0to10.dat')
RawPrediction['ATLAS_5020_R04_Jet_0To10'] = Reader.ReadPrediction('input/STAT2022FirstTry/Prediction_atlas_PbPb5020exponential_inclusive_jet_pt_R0.4_0to10.dat')
RawPrediction['ATLAS_5020_R04_Jet_10To20'] = Reader.ReadPrediction('input/STAT2022FirstTry/Prediction_atlas_PbPb5020exponential_inclusive_jet_pt_R0.4_10to20.dat')
RawPrediction['ATLAS_5020_R04_Jet_20To30'] = Reader.ReadPrediction('input/STAT2022FirstTry/Prediction_atlas_PbPb5020exponential_inclusive_jet_pt_R0.4_20to30.dat')
RawPrediction['ATLAS_5020_R04_Jet_30To40'] = Reader.ReadPrediction('input/STAT2022FirstTry/Prediction_atlas_PbPb5020exponential_inclusive_jet_pt_R0.4_30to40.dat')
RawPrediction['ATLAS_5020_R04_Jet_40To50'] = Reader.ReadPrediction('input/STAT2022FirstTry/Prediction_atlas_PbPb5020exponential_inclusive_jet_pt_R0.4_40to50.dat')
RawPrediction['CMS_5020_R02_Jet_0To10'] = Reader.ReadPrediction('input/STAT2022FirstTry/Prediction_cms_PbPb5020exponential_inclusive_jet_pt_R0.2_0to10.dat')
RawPrediction['CMS_5020_R02_Jet_10To30'] = Reader.ReadPrediction('input/STAT2022FirstTry/Prediction_cms_PbPb5020exponential_inclusive_jet_pt_R0.2_10to30.dat')
RawPrediction['CMS_5020_R02_Jet_30To50'] = Reader.ReadPrediction('input/STAT2022FirstTry/Prediction_cms_PbPb5020exponential_inclusive_jet_pt_R0.2_30to50.dat')
RawPrediction['CMS_5020_R03_Jet_0To10'] = Reader.ReadPrediction('input/STAT2022FirstTry/Prediction_cms_PbPb5020exponential_inclusive_jet_pt_R0.3_0to10.dat')
RawPrediction['CMS_5020_R03_Jet_10To30'] = Reader.ReadPrediction('input/STAT2022FirstTry/Prediction_cms_PbPb5020exponential_inclusive_jet_pt_R0.3_10to30.dat')
RawPrediction['CMS_5020_R03_Jet_30To50'] = Reader.ReadPrediction('input/STAT2022FirstTry/Prediction_cms_PbPb5020exponential_inclusive_jet_pt_R0.3_30to50.dat')
RawPrediction['CMS_5020_R04_Jet_0To10'] = Reader.ReadPrediction('input/STAT2022FirstTry/Prediction_cms_PbPb5020exponential_inclusive_jet_pt_R0.4_0to10.dat')
RawPrediction['CMS_5020_R04_Jet_10To30'] = Reader.ReadPrediction('input/STAT2022FirstTry/Prediction_cms_PbPb5020exponential_inclusive_jet_pt_R0.4_10to30.dat')
RawPrediction['CMS_5020_R04_Jet_30To50'] = Reader.ReadPrediction('input/STAT2022FirstTry/Prediction_cms_PbPb5020exponential_inclusive_jet_pt_R0.4_30to50.dat')
RawPrediction['CMS_5020_R06_Jet_0To10'] = Reader.ReadPrediction('input/STAT2022FirstTry/Prediction_cms_PbPb5020exponential_inclusive_jet_pt_R0.6_0to10.dat')
RawPrediction['CMS_5020_R06_Jet_10To30'] = Reader.ReadPrediction('input/STAT2022FirstTry/Prediction_cms_PbPb5020exponential_inclusive_jet_pt_R0.6_10to30.dat')
RawPrediction['CMS_5020_R06_Jet_30To50'] = Reader.ReadPrediction('input/STAT2022FirstTry/Prediction_cms_PbPb5020exponential_inclusive_jet_pt_R0.6_30to50.dat')
RawPrediction['CMS_5020_R08_Jet_0To10'] = Reader.ReadPrediction('input/STAT2022FirstTry/Prediction_cms_PbPb5020exponential_inclusive_jet_pt_R0.8_0to10.dat')
RawPrediction['CMS_5020_R08_Jet_10To30'] = Reader.ReadPrediction('input/STAT2022FirstTry/Prediction_cms_PbPb5020exponential_inclusive_jet_pt_R0.8_10to30.dat')
RawPrediction['CMS_5020_R08_Jet_30To50'] = Reader.ReadPrediction('input/STAT2022FirstTry/Prediction_cms_PbPb5020exponential_inclusive_jet_pt_R0.8_30to50.dat')
RawPrediction['CMS_5020_R10_Jet_0To10'] = Reader.ReadPrediction('input/STAT2022FirstTry/Prediction_cms_PbPb5020exponential_inclusive_jet_pt_R1.0_0to10.dat')
RawPrediction['CMS_5020_R10_Jet_10To30'] = Reader.ReadPrediction('input/STAT2022FirstTry/Prediction_cms_PbPb5020exponential_inclusive_jet_pt_R1.0_10to30.dat')
RawPrediction['CMS_5020_R10_Jet_30To50'] = Reader.ReadPrediction('input/STAT2022FirstTry/Prediction_cms_PbPb5020exponential_inclusive_jet_pt_R1.0_30to50.dat')



In [3]:
print('Design shape: ' + str(RawDesign['Design'].shape))
for Item in DataList:
    print(Item, RawPrediction[Item]['Prediction'].shape, RawData[Item]['Data']['y'].shape)

Design shape: (40, 6)
PHENIX_200_Hadron_0To10 (60, 15) (15,)
ATLAS_2760_Hadron_0To5 (40, 21) (37,)
CMS_5020_Hadron_30To50 (56, 21) (35,)
STAR_200_R02_Jet_0To10 (60, 9) (9,)
STAR_200_R04_Jet_0To10 (60, 9) (9,)
ALICE_2760_R02_Jet_0To10 (40, 5) (5,)
ATLAS_2760_R04_Jet_0To10 (40, 9) (9,)
CMS_2760_R02_Jet_0To5 (40, 12) (12,)
CMS_2760_R02_Jet_5To10 (40, 11) (11,)
CMS_2760_R03_Jet_0To5 (40, 12) (12,)
CMS_2760_R03_Jet_5To10 (40, 11) (11,)
CMS_2760_R04_Jet_0To5 (40, 12) (12,)
CMS_2760_R04_Jet_5To10 (40, 11) (11,)
ALICE_5020_R02_Jet_0To10 (60, 7) (7,)
ALICE_5020_R04_Jet_0To10 (60, 5) (5,)
ATLAS_5020_R04_Jet_0To10 (60, 15) (15,)
ATLAS_5020_R04_Jet_10To20 (51, 12) (12,)
ATLAS_5020_R04_Jet_20To30 (54, 14) (14,)
ATLAS_5020_R04_Jet_30To40 (55, 13) (13,)
ATLAS_5020_R04_Jet_40To50 (56, 14) (14,)
CMS_5020_R02_Jet_0To10 (60, 3) (3,)
CMS_5020_R02_Jet_10To30 (54, 5) (5,)
CMS_5020_R02_Jet_30To50 (56, 3) (3,)
CMS_5020_R03_Jet_0To10 (60, 3) (3,)
CMS_5020_R03_Jet_10To30 (54, 5) (5,)
CMS_5020_R03_Jet_30To50 (56

In [4]:
NDesign = RawDesign['Design'].shape[0]
for Item in DataList:
    if RawPrediction[Item]['Prediction'].shape[0] > NDesign:
        # print(RawPrediction[Item]['Prediction'].shape)
        ToDelete = RawPrediction[Item]['Prediction'].shape[0] - NDesign
        RawPrediction[Item]['Prediction'] = np.delete(RawPrediction[Item]['Prediction'], range(NDesign, RawPrediction[Item]['Prediction'].shape[0]), axis = 0)
        # print(RawPrediction[Item]['Prediction'].shape)

RawDesign['Design'] = np.delete(RawDesign['Design'], [18, 27, 37, 39], axis = 0)
for Item in DataList:
    RawPrediction[Item]['Prediction'] = np.delete(RawPrediction[Item]['Prediction'], [18, 27, 37, 39], axis = 0)
    


In [5]:
def DeleteRawData(RawDataObject, Items):
    RawDataObject["Data"]["x"]    = np.delete(RawDataObject["Data"]["x"], Items, axis = 0)
    RawDataObject["Data"]["xerr"] = np.delete(RawDataObject["Data"]["xerr"], Items, axis = 0)
    RawDataObject["Data"]["y"]    = np.delete(RawDataObject["Data"]["y"], Items, axis = 0)
    for item in RawDataObject["Data"]["yerr"]:
        RawDataObject["Data"]["yerr"][item] = np.delete(RawDataObject["Data"]["yerr"][item], Items, axis = 0)

In [6]:
DeleteRawData(RawData["STAR_200_R02_Jet_0To10"], [0, 1, 2])
RawPrediction["STAR_200_R02_Jet_0To10"]["Prediction"] = np.delete(RawPrediction["STAR_200_R02_Jet_0To10"]["Prediction"], [0, 1, 2], axis = 1)

DeleteRawData(RawData["STAR_200_R04_Jet_0To10"], [0, 1, 2])
RawPrediction["STAR_200_R04_Jet_0To10"]["Prediction"] = np.delete(RawPrediction["STAR_200_R04_Jet_0To10"]["Prediction"], [0, 1, 2], axis = 1)

DeleteRawData(RawData["ATLAS_2760_Hadron_0To5"], range(0, 16))
DeleteRawData(RawData["CMS_5020_Hadron_30To50"], range(0, 14))

In [7]:
# for Item in DataList:
#     RawPrediction[Item]['Prediction'] = np.minimum(RawPrediction[Item]['Prediction'], 1.5)

In [8]:
print('Design shape: ' + str(RawDesign['Design'].shape))
for Item in DataList:
    print(Item, RawPrediction[Item]['Prediction'].shape, RawData[Item]['Data']['y'].shape)

Design shape: (36, 6)
PHENIX_200_Hadron_0To10 (36, 15) (15,)
ATLAS_2760_Hadron_0To5 (36, 21) (21,)
CMS_5020_Hadron_30To50 (36, 21) (21,)
STAR_200_R02_Jet_0To10 (36, 6) (6,)
STAR_200_R04_Jet_0To10 (36, 6) (6,)
ALICE_2760_R02_Jet_0To10 (36, 5) (5,)
ATLAS_2760_R04_Jet_0To10 (36, 9) (9,)
CMS_2760_R02_Jet_0To5 (36, 12) (12,)
CMS_2760_R02_Jet_5To10 (36, 11) (11,)
CMS_2760_R03_Jet_0To5 (36, 12) (12,)
CMS_2760_R03_Jet_5To10 (36, 11) (11,)
CMS_2760_R04_Jet_0To5 (36, 12) (12,)
CMS_2760_R04_Jet_5To10 (36, 11) (11,)
ALICE_5020_R02_Jet_0To10 (36, 7) (7,)
ALICE_5020_R04_Jet_0To10 (36, 5) (5,)
ATLAS_5020_R04_Jet_0To10 (36, 15) (15,)
ATLAS_5020_R04_Jet_10To20 (36, 12) (12,)
ATLAS_5020_R04_Jet_20To30 (36, 14) (14,)
ATLAS_5020_R04_Jet_30To40 (36, 13) (13,)
ATLAS_5020_R04_Jet_40To50 (36, 14) (14,)
CMS_5020_R02_Jet_0To10 (36, 3) (3,)
CMS_5020_R02_Jet_10To30 (36, 5) (5,)
CMS_5020_R02_Jet_30To50 (36, 3) (3,)
CMS_5020_R03_Jet_0To10 (36, 3) (3,)
CMS_5020_R03_Jet_10To30 (36, 5) (5,)
CMS_5020_R03_Jet_30To50 (36

### Run this block for RHIC + LHC

In [9]:
# All
# IncludeDataList = ['PHENIX_200_Hadron_0To10', 'ATLAS_2760_Hadron_0To5', 'CMS_5020_Hadron_30To50',
#            'STAR_200_R02_Jet_0To10', 'STAR_200_R04_Jet_0To10',
#            'ALICE_2760_R02_Jet_0To10', 'ATLAS_2760_R04_Jet_0To10',
#            'CMS_2760_R02_Jet_0To5', 'CMS_2760_R02_Jet_5To10',
#            'CMS_2760_R03_Jet_0To5', 'CMS_2760_R03_Jet_5To10',
#            'CMS_2760_R04_Jet_0To5', 'CMS_2760_R04_Jet_5To10',
#            'ALICE_5020_R02_Jet_0To10', 'ALICE_5020_R04_Jet_0To10',
#            'ATLAS_5020_R04_Jet_0To10', 'ATLAS_5020_R04_Jet_10To20', 'ATLAS_5020_R04_Jet_20To30',
#            'ATLAS_5020_R04_Jet_30To40', 'ATLAS_5020_R04_Jet_40To50']

# 200 GeV only
# IncludeDataList = ['PHENIX_200_Hadron_0To10', 'STAR_200_R02_Jet_0To10', 'STAR_200_R04_Jet_0To10']

# No 5.02 TeV jets
# IncludeDataList = ['PHENIX_200_Hadron_0To10', 'STAR_200_R02_Jet_0To10', 'STAR_200_R04_Jet_0To10',
#            'ALICE_2760_R02_Jet_0To10', 'ATLAS_2760_R04_Jet_0To10',
#            'CMS_2760_R02_Jet_0To5', 'CMS_2760_R02_Jet_5To10',
#            'CMS_2760_R03_Jet_0To5', 'CMS_2760_R03_Jet_5To10',
#            'CMS_2760_R04_Jet_0To5', 'CMS_2760_R04_Jet_5To10']

# Functional
IncludeDataList = ['PHENIX_200_Hadron_0To10', 'ATLAS_2760_Hadron_0To5',
           'STAR_200_R02_Jet_0To10', 'STAR_200_R04_Jet_0To10',
           'ALICE_2760_R02_Jet_0To10', 'ATLAS_2760_R04_Jet_0To10',
           'CMS_2760_R02_Jet_0To5', 'CMS_2760_R02_Jet_5To10',
           'CMS_2760_R03_Jet_0To5', 'CMS_2760_R03_Jet_5To10',
           'CMS_2760_R04_Jet_0To5', 'CMS_2760_R04_Jet_5To10',
           'ALICE_5020_R02_Jet_0To10', 'ALICE_5020_R04_Jet_0To10',
           'ATLAS_5020_R04_Jet_0To10', 'ATLAS_5020_R04_Jet_20To30',
           'ATLAS_5020_R04_Jet_30To40', 'ATLAS_5020_R04_Jet_40To50',
           'CMS_5020_R02_Jet_0To10', 'CMS_5020_R02_Jet_10To30', 'CMS_5020_R02_Jet_30To50',
           'CMS_5020_R03_Jet_0To10', 'CMS_5020_R03_Jet_10To30', 'CMS_5020_R03_Jet_30To50',
           'CMS_5020_R04_Jet_0To10', 'CMS_5020_R04_Jet_10To30', 'CMS_5020_R04_Jet_30To50',
           'CMS_5020_R06_Jet_0To10', 'CMS_5020_R06_Jet_10To30', 'CMS_5020_R06_Jet_30To50',
           'CMS_5020_R08_Jet_0To10', 'CMS_5020_R08_Jet_10To30', 'CMS_5020_R08_Jet_30To50',
           'CMS_5020_R10_Jet_10To30', 'CMS_5020_R10_Jet_30To50']

# Central only
IncludeDataList = ['PHENIX_200_Hadron_0To10', 'ATLAS_2760_Hadron_0To5',
           'STAR_200_R02_Jet_0To10', 'STAR_200_R04_Jet_0To10',
           'ALICE_2760_R02_Jet_0To10', 'ATLAS_2760_R04_Jet_0To10',
           'CMS_2760_R02_Jet_0To5', 'CMS_2760_R02_Jet_5To10',
           'CMS_2760_R03_Jet_0To5', 'CMS_2760_R03_Jet_5To10',
           'CMS_2760_R04_Jet_0To5', 'CMS_2760_R04_Jet_5To10',
           'ALICE_5020_R02_Jet_0To10', 'ALICE_5020_R04_Jet_0To10',
           'ATLAS_5020_R04_Jet_0To10',
           'CMS_5020_R02_Jet_0To10',
           'CMS_5020_R03_Jet_0To10',
           'CMS_5020_R04_Jet_0To10',
           'CMS_5020_R06_Jet_0To10',
           'CMS_5020_R08_Jet_0To10']



# Initialize empty dictionary
AllData = {}

# Basic information
AllData["systems"] = ["HeavyIon"]
AllData["keys"] = RawDesign["Parameter"]
AllData["labels"] = RawDesign["Parameter"]
AllData["ranges"] = [(0.1, 0.5), (1, 10), (np.log(0.005), np.log(10)), (np.log(0.005), np.log(10)), (0, 1.5), (np.log(0.05), np.log(100))]
AllData["observables"] = [('R_AA', IncludeDataList)]

AllData["labels"][2] = 'log(' + AllData["labels"][2] + ')'
AllData["labels"][3] = 'log(' + AllData["labels"][3] + ')'
AllData["labels"][5] = 'log(' + AllData["labels"][5] + ')'

# Data points
Data = {"HeavyIon": {"R_AA": {}}}
for Item in IncludeDataList:
    Data["HeavyIon"]["R_AA"][Item] = RawData[Item]["Data"]

# Model predictions
Prediction = {"HeavyIon": {"R_AA": {}}}
for Item in IncludeDataList:
    Prediction["HeavyIon"]["R_AA"][Item] = {}
    Prediction["HeavyIon"]["R_AA"][Item]["Y"] = RawPrediction[Item]["Prediction"]
    Prediction["HeavyIon"]["R_AA"][Item]["x"] = RawData[Item]["Data"]['x']

# Covariance matrices - the indices are [system][measurement1][measurement2], each one is a block of matrix
Covariance = Reader.InitializeCovariance(Data)
for Item in IncludeDataList:
    # print(Item)
    Covariance["HeavyIon"][("R_AA", Item)][("R_AA", Item)] = Reader.EstimateCovariance(RawData[Item], RawData[Item], SysLength = {"default": 0.10})

# Assign data to the dictionary
AllData["design"] = RawDesign["Design"]
AllData["model"] = Prediction
AllData["data"] = Data
AllData["cov"] = Covariance

# Save to the desired pickle file
with open('input/default.p', 'wb') as handle:
    pickle.dump(AllData, handle, protocol = pickle.HIGHEST_PROTOCOL)

In [10]:
#

In [11]:
# This block is backup from previous iteration

# Initialize empty dictionary
# AllData = {}

# Basic information
# AllData["systems"] = ["AuAu200", "PbPb2760", "PbPb5020"]
# AllData["keys"] = RawDesign["Parameter"]
# AllData["labels"] = RawDesign["Parameter"]
# AllData["ranges"] = [(0, 1.5), (0, 1.0), (0, 20), (0, 20), (1, 4)]
# AllData["observables"] = [('R_AA', ['C0', 'C1'])]

# Data points
# Data = {"AuAu200": {"R_AA": {"C0": RawData1["Data"], "C1": RawData2["Data"]}},
#     "PbPb2760": {"R_AA": {"C0": RawData3["Data"], "C1": RawData4["Data"]}},
#     "PbPb5020": {"R_AA": {"C0": RawData5["Data"], "C1": RawData6["Data"]}}}

# Model predictions
# Prediction = {"AuAu200": {"R_AA": {"C0": {"Y": RawPrediction1["Prediction"], "x": RawData1["Data"]['x']},
#                                    "C1": {"Y": RawPrediction2["Prediction"], "x": RawData2["Data"]['x']}}},
#              "PbPb2760": {"R_AA": {"C0": {"Y": RawPrediction3["Prediction"], "x": RawData3["Data"]['x']},
#                                    "C1": {"Y": RawPrediction4["Prediction"], "x": RawData4["Data"]['x']}}},
#              "PbPb5020": {"R_AA": {"C0": {"Y": RawPrediction5["Prediction"], "x": RawData5["Data"]['x']},
#                                    "C1": {"Y": RawPrediction6["Prediction"], "x": RawData6["Data"]['x']}}}}

# Covariance matrices - the indices are [system][measurement1][measurement2], each one is a block of matrix
# Covariance = Reader.InitializeCovariance(Data)
# Covariance["AuAu200"][("R_AA", "C0")][("R_AA", "C0")] = Reader.EstimateCovariance(RawData1, RawData1, SysLength = {"default": 0.05})
# Covariance["AuAu200"][("R_AA", "C1")][("R_AA", "C1")] = Reader.EstimateCovariance(RawData2, RawData2, SysLength = {"default": 0.10})
# Covariance["PbPb2760"][("R_AA", "C0")][("R_AA", "C0")] = Reader.EstimateCovariance(RawData3, RawData3, SysLength = {"default": 0.15})
# Covariance["PbPb2760"][("R_AA", "C1")][("R_AA", "C1")] = Reader.EstimateCovariance(RawData4, RawData4, SysLength = {"default": 0.20})
# Covariance["PbPb5020"][("R_AA", "C0")][("R_AA", "C0")] = Reader.EstimateCovariance(RawData5, RawData5, SysLength = {"default": 0.25})
# Covariance["PbPb5020"][("R_AA", "C1")][("R_AA", "C1")] = Reader.EstimateCovariance(RawData6, RawData6, SysLength = {"default": 0.30})

# This is how we can add off-diagonal matrices
# Covariance["PbPb5020"][("R_AA", "C0")][("R_AA", "C1")] = Reader.EstimateCovariance(RawData5, RawData6, SysLength = {"default": 100}, SysStrength = {"default": 0.1})
# Covariance["PbPb5020"][("R_AA", "C1")][("R_AA", "C0")] = Reader.EstimateCovariance(RawData6, RawData5, SysLength = {"default": 100}, SysStrength = {"default": 0.1})

# This is how we can supply external pre-generated matrices
# Covariance["AuAu200"][("R_AA", "C0")][("R_AA", "C0")] = RawCov1["Matrix"]


# Assign data to the dictionary
# AllData["design"] = RawDesign["Design"]
# AllData["model"] = Prediction
# AllData["data"] = Data
# AllData["cov"] = Covariance

# Save to the desired pickle file
# with open('input/default.p', 'wb') as handle:
#     pickle.dump(AllData, handle, protocol = pickle.HIGHEST_PROTOCOL)

### Optional: clean past files

In [12]:
# Clean past MCMC samples
if os.path.exists('cache/mcmc_chain.hdf'):
    os.remove("cache/mcmc_chain.hdf")

# Clean past emulator
for system in AllData["systems"]:
    if os.path.exists('cache/emulator/' + system + ".pkl"):
        os.remove('cache/emulator/' + system + ".pkl")

## Step 2: run emulator

In [13]:
! python3 -m src.emulator --retrain --npc 24 --kernelchoice MaternNoise

[INFO][emulator] training emulator for system HeavyIon (24 PC, 0 restarts, alpha=0.00, kernel=MaternNoise, noise=-1.000000)
HeavyIon
None
[0.4        9.         7.60090246 7.60090246 1.5        7.60090246]
[INFO][emulator] writing cache file cache/emulator/HeavyIon.pkl
HeavyIon
24 PCs explain 0.99456 of variance
GP 0: 0.84952 of variance, LML = 11.125, kernel: Matern(length_scale=[0.235, 10.7, 76, 76, 5.47, 76], nu=2.5) + WhiteKernel(noise_level=0.00185)
GP 1: 0.03722 of variance, LML = -48.796, kernel: Matern(length_scale=[0.04, 1.19, 76, 2.22, 0.483, 76], nu=2.5) + WhiteKernel(noise_level=0.0001)
GP 2: 0.01702 of variance, LML = -48.6, kernel: Matern(length_scale=[0.0605, 1.06, 76, 2.22, 0.57, 76], nu=2.5) + WhiteKernel(noise_level=0.0001)
GP 3: 0.01675 of variance, LML = -48.405, kernel: Matern(length_scale=[4, 1.27, 2.57, 1.23, 15, 4.77], nu=2.5) + WhiteKernel(noise_level=0.000138)
GP 4: 0.00955 of variance, LML = -48.04, kernel: Matern(length_scale=[4, 4.51, 76, 13.1, 0.176, 0.76]

In [14]:
from src import lazydict, emulator
Emulator = emulator.Emulator.from_cache('HeavyIon')

## Step 3: MCMC sampling

In [ ]:
if os.path.exists('cache/mcmc_chain.hdf'):
    os.remove("cache/mcmc_chain.hdf")
! python3 -m src.mcmc --nwalkers 100 --nburnsteps 200 200

HeavyIon
None
[INFO][mcmc] no existing chain found, starting initial burn-in
[INFO][mcmc] step 10: acceptance fraction: mean 0.3070, std 0.1756, min 0.0000, max 0.7000
[INFO][mcmc] step 20: acceptance fraction: mean 0.2895, std 0.1258, min 0.0000, max 0.6000
[INFO][mcmc] step 30: acceptance fraction: mean 0.2833, std 0.1092, min 0.0333, max 0.5000
[INFO][mcmc] step 40: acceptance fraction: mean 0.2792, std 0.1010, min 0.0500, max 0.5000
[INFO][mcmc] step 50: acceptance fraction: mean 0.2792, std 0.0976, min 0.0400, max 0.5000
[INFO][mcmc] step 60: acceptance fraction: mean 0.2743, std 0.0935, min 0.0333, max 0.4667
[INFO][mcmc] step 70: acceptance fraction: mean 0.2747, std 0.0902, min 0.0429, max 0.4857
[INFO][mcmc] step 80: acceptance fraction: mean 0.2735, std 0.0905, min 0.0375, max 0.5000
[INFO][mcmc] step 90: acceptance fraction: mean 0.2734, std 0.0890, min 0.0333, max 0.4667
[INFO][mcmc] step 100: acceptance fraction: mean 0.2729, std 0.0856, min 0.0300, max 0.4400
[INFO][mcmc]

## Step 4: Analyze posterior samples

In [ ]:
import src
src.Initialize()
from src import mcmc
chain = mcmc.Chain()
MCMCSamples = chain.load()

# TransformedSamples = np.copy(MCMCSamples)
# TransformedSamples[:,0] = MCMCSamples[:,0] * MCMCSamples[:,1]
# TransformedSamples[:,1] = MCMCSamples[:,0] - MCMCSamples[:,0] * MCMCSamples[:,1]
# TransformedSamples[:,2] = MCMCSamples[:,2]
# TransformedSamples[:,3] = MCMCSamples[:,3]
# TransformedSamples[:,4] = MCMCSamples[:,4]

In [ ]:
# ! python3 -m src.plots posterior gp diag_emu

## Step 5: adding all sorts of plots

In [ ]:
with chain.dataset() as d:
    W = d.shape[0]     # number of walkers
    S = d.shape[1]     # number of steps
    N = d.shape[2]     # number of paramters
    T = int(S / 100)   # "thinning"
    A = 20 / W
    figure, axes = plt.subplots(figsize = (15, 2 * N), ncols = 1, nrows = N)
    for i, ax in enumerate(axes):
        for j in range(0, W):
            ax.plot(range(0, S, T), d[j, ::T, i], alpha = A)
    plt.tight_layout(True)
    plt.savefig('plots/MCMCSamples.pdf', dpi = 192)
    plt.savefig('plots/MCMCSamples.png', dpi = 192)

In [ ]:
chain.dataset()

In [ ]:
NDimension = len(AllData["labels"])
Ranges = np.array(AllData["ranges"]).T
figure, axes = plt.subplots(figsize = (3 * NDimension, 3 * NDimension), ncols = NDimension, nrows = NDimension)
Names = AllData["labels"]
for i, row in enumerate(axes):
    for j, ax in enumerate(row):
        if i==j:
            ax.hist(MCMCSamples[:,i], bins=50,
                    range=Ranges[:,i], histtype='step', color='green')
            ax.set_xlabel(Names[i])
            ax.set_xlim(*Ranges[:,j])
        if i>j:
            ax.hist2d(MCMCSamples[:, j], MCMCSamples[:, i], 
                      bins=50, range=[Ranges[:,j], Ranges[:,i]], 
                      cmap='Greens')
            ax.set_xlabel(Names[j])
            ax.set_ylabel(Names[i])
            ax.set_xlim(*Ranges[:,j])
            ax.set_ylim(*Ranges[:,i])
        if i<j:
            ax.axis('off')
plt.tight_layout(True)
plt.savefig('plots/Correlation.pdf', dpi = 192)
plt.savefig('plots/Correlation.png', dpi = 192)
# figure

In [ ]:
NDimension = len(AllData["labels"])
Ranges = np.array(AllData["ranges"]).T
DesignPoints = AllData["design"]

figure, axes = plt.subplots(figsize = (3 * NDimension, 3 * NDimension), ncols = NDimension, nrows = NDimension)
Names = AllData["labels"]
for i, row in enumerate(axes):
    for j, ax in enumerate(row):
        if i>j:
            ax.hist2d(DesignPoints[:, j], DesignPoints[:, i], 
                      bins=50, range=[Ranges[:,j], Ranges[:,i]], 
                      cmap='Greens')
            ax.set_xlabel(Names[j])
            ax.set_ylabel(Names[i])
            ax.set_xlim(*Ranges[:,j])
            ax.set_ylim(*Ranges[:,i])
        if i<=j:
            ax.axis('off')
plt.tight_layout(True)
plt.savefig('plots/DesignPoints.pdf', dpi = 192)
# figure

In [ ]:
Examples = MCMCSamples[ np.random.choice(range(len(MCMCSamples)), 500), :]

TempPrediction = {"HeavyIon": Emulator.predict(Examples)}

SystemCount = len(AllData["systems"])
BinCount = len(AllData['observables'][0][1])

RC = 5
CC = int(np.ceil(BinCount / RC))

figure, axes = plt.subplots(figsize = (3 * CC, 3 * RC), nrows = RC, ncols = CC)

for s2 in range(0, BinCount):
    ax = s2 % RC
    ay = int(np.floor(s2 / RC))
    axes[ax][ay].set_xlabel(r"$p_{T}$")
    axes[ax][ay].set_ylabel(r"$R_{AA}$")
        
    S1 = AllData["systems"][0]
    O  = AllData["observables"][0][0]
    S2 = AllData["observables"][0][1][s2]
        
    DX = AllData["data"][S1][O][S2]['x']
    DY = AllData["data"][S1][O][S2]['y']
    DE = np.sqrt(AllData["data"][S1][O][S2]['yerr']['stat'][:,0]**2 + AllData["data"][S1][O][S2]['yerr']['sys'][:,0]**2)
                
    axes[ax][ay].set_title(AllData["observables"][0][1][s2])

    for i, y in enumerate(TempPrediction[S1][O][S2]):
        axes[ax][ay].plot(DX, y, 'b-', alpha=0.05, label="Posterior" if i==0 else '')
    axes[ax][ay].errorbar(DX, DY, yerr = DE, fmt='ro', label="Measurements")

plt.tight_layout(True)
figure.savefig('plots/ObservablePosterior.pdf', dpi = 192)
figure.savefig('plots/ObservablePosterior.png', dpi = 192)
# figure

In [ ]:
Examples = AllData["design"]

TempPrediction = {"HeavyIon": Emulator.predict(Examples)}

SystemCount = len(AllData["systems"])
BinCount = len(AllData['observables'][0][1])

RC = 5
CC = int(np.ceil(BinCount / RC))

figure, axes = plt.subplots(figsize = (3 * CC, 3 * RC), nrows = RC, ncols = CC)

for s2 in range(0, BinCount):
    ax = s2 % RC
    ay = int(np.floor(s2 / RC))
    axes[ax][ay].set_xlabel(r"$p_{T}$")
    axes[ax][ay].set_ylabel(r"$R_{AA}$")
        
    S1 = AllData["systems"][0]
    O  = AllData["observables"][0][0]
    S2 = AllData["observables"][0][1][s2]
        
    DX = AllData["data"][S1][O][S2]['x']
    DY = AllData["data"][S1][O][S2]['y']
    DE = np.sqrt(AllData["data"][S1][O][S2]['yerr']['stat'][:,0]**2 + AllData["data"][S1][O][S2]['yerr']['sys'][:,0]**2)
                
    axes[ax][ay].set_title(AllData["observables"][0][1][s2])
    
    for i, y in enumerate(TempPrediction[S1][O][S2]):
        axes[ax][ay].plot(DX, y, 'b-', alpha=0.2, label="Posterior" if i==0 else '')
    axes[ax][ay].errorbar(DX, DY, yerr = DE, fmt='ro', label="Measurements")

plt.tight_layout(True)
figure.savefig('plots/ObservableDesign.pdf', dpi = 192)
figure.savefig('plots/ObservableDesign.png', dpi = 192)
# figure

In [ ]:
TempPrediction = AllData["model"]

SystemCount = len(AllData["systems"])
BinCount = len(AllData['observables'][0][1])

RC = 5
CC = int(np.ceil(BinCount / RC))

figure, axes = plt.subplots(figsize = (3 * CC, 3 * RC), nrows = RC, ncols = CC)

for s2 in range(0, BinCount):
    ax = s2 % RC
    ay = int(np.floor(s2 / RC))
    axes[ax][ay].set_xlabel(r"$p_{T}$")
    axes[ax][ay].set_ylabel(r"$R_{AA}$")
        
    S1 = AllData["systems"][0]
    O  = AllData["observables"][0][0]
    S2 = AllData["observables"][0][1][s2]
        
    DX = AllData["data"][S1][O][S2]['x']
    DY = AllData["data"][S1][O][S2]['y']
    DE = np.sqrt(AllData["data"][S1][O][S2]['yerr']['stat'][:,0]**2 + AllData["data"][S1][O][S2]['yerr']['sys'][:,0]**2)

    axes[ax][ay].set_title(AllData["observables"][0][1][s2])
                
    for i, y in enumerate(TempPrediction[S1][O][S2]['Y']):
        axes[ax][ay].plot(DX, y, 'b-', alpha=0.2, label="Posterior" if i==0 else '')
    axes[ax][ay].errorbar(DX, DY, yerr = DE, fmt='ro', label="Measurements")

plt.tight_layout(True)
figure.savefig('plots/Design.pdf', dpi = 192)
figure.savefig('plots/Design.png', dpi = 192)
# figure

In [ ]:
# close all plots to save memory
plt.close('all')